In [1]:
# 03_sentiment.ipynb
# Collect real news via NewsAPI + FinBERT sentiment analysis

import requests
import pandas as pd
import os
from dotenv import load_dotenv
from transformers import pipeline

env_path = os.path.expanduser("~/Desktop/SentriVaR-500/.env")
load_dotenv(env_path)
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

# Load FinBERT
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

print("Loaded")
print(f"API key check: {NEWSAPI_KEY[:8]}...")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded
API key check: 243b95d4...


In [2]:
# Fetch real news headlines via NewsAPI
def fetch_news(ticker, api_key, days_back=30):
    """
    Fetch news headlines for a given ticker/query.
    - ticker: search term (e.g. "Apple")
    - days_back: how many days of news to pull
    """
    from datetime import datetime, timedelta

    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": ticker,
        "from": start_date,
        "to": end_date,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 50,
        "apiKey": api_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    if data["status"] != "ok":
        print(f"Error: {data.get('message')}")
        return []

    headlines = [article["title"] for article in data["articles"]]
    print(f"Fetched {len(headlines)} headlines for {ticker}")
    return headlines

# Test
apple_news = fetch_news("Apple stock", NEWSAPI_KEY, days_back=30)
print("\nFirst 3 headlines:")
for h in apple_news[:3]:
    print(f"  - {h}")

Fetched 50 headlines for Apple stock

First 3 headlines:
  - Stocks priced for ‘sunshine and rainbows’ now face earnings test
  - Investing in Kids and Country
  - Julian Alvarez’s World Cup heroics spotlight Polymarket’s growing sports betting volumes


In [3]:
# Search queries per ticker, for more precise results
TICKER_QUERIES = {
    "AAPL":  "Apple Inc stock market",
    "MSFT":  "Microsoft stock market",
    "GOOGL": "Google Alphabet stock market",
    "JPM":   "JPMorgan Chase stock market",
    "SOXX":  "semiconductor ETF iShares stock market"
}

# Fetch news for every ticker
all_news = {}
for ticker, query in TICKER_QUERIES.items():
    headlines = fetch_news(query, NEWSAPI_KEY, days_back=30)
    all_news[ticker] = headlines

print("\nDone:")
for ticker, headlines in all_news.items():
    print(f"  {ticker}: {len(headlines)}")

Fetched 50 headlines for Apple Inc stock market
Fetched 50 headlines for Microsoft stock market
Fetched 49 headlines for Google Alphabet stock market
Fetched 48 headlines for JPMorgan Chase stock market
Fetched 50 headlines for semiconductor ETF iShares stock market

Done:
  AAPL: 50
  MSFT: 50
  GOOGL: 49
  JPM: 48
  SOXX: 50


In [4]:
# Sentiment scoring with FinBERT
def analyze_sentiment(headlines):
    """
    Score a list of headlines.
    - positive: +1, negative: -1, neutral: 0
    - returns average score, range -1 to +1
    """
    if not headlines:
        return 0.0

    score_map = {"positive": 1, "negative": -1, "neutral": 0}
    scores = []
    for headline in headlines:
        result = nlp(headline[:512])[0]  # 512 char limit
        score = score_map[result["label"]] * result["score"]
        scores.append(score)

    return round(sum(scores) / len(scores), 4)

# Score every ticker
print("Running sentiment analysis... (takes 1-2 min)")
sentiment_scores = {}
for ticker, headlines in all_news.items():
    sentiment_scores[ticker] = analyze_sentiment(headlines)
    print(f"  {ticker}: {sentiment_scores[ticker]}")

print("\nDone!")

Running sentiment analysis... (takes 1-2 min)
  AAPL: -0.1495
  MSFT: 0.042
  GOOGL: -0.2055
  JPM: -0.1315
  SOXX: -0.0922

Done!


In [5]:
# Save results
data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
from datetime import datetime

sentiment_df = pd.DataFrame([{
    "date": datetime.today().strftime("%Y-%m-%d"),
    "ticker": ticker,
    "sentiment_score": score
} for ticker, score in sentiment_scores.items()])

sentiment_df.to_csv(f"{data_path}/sentiment_scores.csv", index=False)
print("Saved: sentiment_scores.csv")
print(sentiment_df)

Saved: sentiment_scores.csv
         date ticker  sentiment_score
0  2026-07-13   AAPL          -0.1495
1  2026-07-13   MSFT           0.0420
2  2026-07-13  GOOGL          -0.2055
3  2026-07-13    JPM          -0.1315
4  2026-07-13   SOXX          -0.0922


In [6]:
# News classifier: split headlines into systemic vs idiosyncratic
SYSTEMIC_KEYWORDS = [
    "federal reserve", "fed", "interest rate", "inflation", "gdp",
    "recession", "tariff", "trade war", "unemployment", "central bank",
    "treasury", "yield", "macro", "economy", "market crash"
]

IDIOSYNCRATIC_KEYWORDS = {
    "AAPL":  ["apple", "iphone", "tim cook", "app store", "mac"],
    "MSFT":  ["microsoft", "azure", "satya nadella", "windows", "copilot"],
    "GOOGL": ["google", "alphabet", "youtube", "sundar pichai", "gemini"],
    "JPM":   ["jpmorgan", "jamie dimon", "chase", "investment bank"],
    "SOXX":  ["semiconductor", "nvidia", "amd", "intel", "chips act", "tsmc"]
}

def classify_news(headline):
    """Classify a headline as Systemic / Idiosyncratic / Unrelated."""
    headline_lower = headline.lower()

    if any(kw in headline_lower for kw in SYSTEMIC_KEYWORDS):
        return "systemic"

    for ticker, keywords in IDIOSYNCRATIC_KEYWORDS.items():
        if any(kw in headline_lower for kw in keywords):
            return f"idiosyncratic_{ticker}"

    return "unrelated"

# Test
test_headlines = [
    "Federal Reserve signals interest rate cuts in 2026",
    "Apple reports record iPhone sales in Q2",
    "JPMorgan warns of recession risk amid tariff uncertainty",
    "Google launches new Gemini AI model",
    "Local restaurant opens in downtown Manhattan"
]
for h in test_headlines:
    print(f"{classify_news(h):25s} → {h}")

systemic                  → Federal Reserve signals interest rate cuts in 2026
idiosyncratic_AAPL        → Apple reports record iPhone sales in Q2
systemic                  → JPMorgan warns of recession risk amid tariff uncertainty
idiosyncratic_GOOGL       → Google launches new Gemini AI model
unrelated                 → Local restaurant opens in downtown Manhattan


In [7]:
# Fetch, classify, and score all news in one pass
TICKER_QUERIES = {
    "AAPL":  "Apple Inc stock market",
    "MSFT":  "Microsoft stock market",
    "GOOGL": "Google Alphabet stock market",
    "JPM":   "JPMorgan Chase stock market",
    "SOXX":  "SOXX stock market"
}

# Macro news gets its own search
SYSTEMIC_QUERY = "federal reserve inflation recession tariff economy 2026"

def fetch_and_classify(query, api_key, days_back=30):
    """Fetch headlines, classify them, and score sentiment in one step."""
    from datetime import datetime, timedelta

    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "from": start_date,
        "to": end_date,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 50,
        "apiKey": api_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    if data["status"] != "ok":
        print(f"Error: {data.get('message')}")
        return []

    results = []
    for article in data["articles"]:
        headline = article["title"]
        category = classify_news(headline)
        sentiment = nlp(headline[:512])[0]
        score_map = {"positive": 1, "negative": -1, "neutral": 0}
        sentiment_score = score_map[sentiment["label"]] * sentiment["score"]

        results.append({
            "headline": headline,
            "category": category,
            "sentiment_label": sentiment["label"],
            "sentiment_score": round(sentiment_score, 4),
            "published_at": article["publishedAt"]
        })

    return results

# Fetch macro news
print("Fetching macro news...")
systemic_news = fetch_and_classify(SYSTEMIC_QUERY, NEWSAPI_KEY)
print(f"  Fetched: {len(systemic_news)}")

# Fetch per-ticker news
all_news = {}
for ticker, query in TICKER_QUERIES.items():
    print(f"Fetching news for {ticker}...")
    all_news[ticker] = fetch_and_classify(query, NEWSAPI_KEY)
    print(f"  Fetched: {len(all_news[ticker])}")

print("\nDone!")

Fetching macro news...
  Fetched: 2
Fetching news for AAPL...
  Fetched: 50
Fetching news for MSFT...
  Fetched: 50
Fetching news for GOOGL...
  Fetched: 49
Fetching news for JPM...
  Fetched: 48
Fetching news for SOXX...
  Fetched: 50

Done!


In [9]:
# Only 3 macro headlines is too few → split into multiple queries
SYSTEMIC_QUERIES = [
    "federal reserve interest rate 2026",
    "inflation economy recession 2026",
    "tariff trade war market 2026"
]

systemic_news = []
for query in SYSTEMIC_QUERIES:
    news = fetch_and_classify(query, NEWSAPI_KEY)
    systemic_news.extend(news)
    print(f"  '{query}': {len(news)}")

# Drop duplicates
systemic_df = pd.DataFrame(systemic_news).drop_duplicates(subset=["headline"])
print(f"\nTotal macro headlines (deduped): {len(systemic_df)}")
systemic_df.head()

  'federal reserve interest rate 2026': 49
  'inflation economy recession 2026': 47
  'tariff trade war market 2026': 49

Total macro headlines (deduped): 139


,headline,category,sentiment_label,sentiment_score,published_at
0,The Fed’s new ‘brains trust’ shows new chair m...,systemic,neutral,0.0000,2026-07-12T07:50:35Z
1,3 reasons Bitcoin is stuck in a bear market—an...,unrelated,negative,-0.7136,2026-07-12T07:00:00Z
2,Stocks priced for ‘sunshine and rainbows’ now ...,unrelated,negative,-0.9355,2026-07-12T05:12:09Z
3,The Sustainability of AI Investment amidst Hig...,systemic,neutral,0.0000,2026-07-12T05:06:53Z
4,Fed officials divided on US inflation views; U...,systemic,negative,-0.7814,2026-07-12T04:43:50Z


In [10]:
# Organize per-ticker news into DataFrames
idiosyncratic_dfs = {}
for ticker, news in all_news.items():
    df = pd.DataFrame(news).drop_duplicates(subset=["headline"])
    # Keep only idiosyncratic headlines
    idio_df = df[df["category"] == f"idiosyncratic_{ticker}"]
    idiosyncratic_dfs[ticker] = idio_df
    print(f"{ticker}: {len(df)} total → {len(idio_df)} idiosyncratic")

# Filter macro headlines
systemic_only = systemic_df[systemic_df["category"] == "systemic"]
print(f"\nSystemic headlines: {len(systemic_only)}")

AAPL: 49 total → 7 idiosyncratic
MSFT: 49 total → 5 idiosyncratic
GOOGL: 47 total → 12 idiosyncratic
JPM: 46 total → 9 idiosyncratic
SOXX: 49 total → 8 idiosyncratic

Systemic headlines: 40


In [11]:
# Calculate split sentiment scores
def get_sentiment_score(df):
    """Average sentiment score for a DataFrame of headlines."""
    if df.empty:
        return 0.0
    return round(df["sentiment_score"].mean(), 4)

# Macro sentiment (feeds into regime detection)
systemic_score = get_sentiment_score(systemic_only)
print(f"Systemic sentiment score: {systemic_score:.4f}")
print(f"  → {'negative' if systemic_score < 0 else 'positive'} macro backdrop\n")

# Per-ticker idiosyncratic sentiment (feeds into individual risk weighting)
idiosyncratic_scores = {}
for ticker, df in idiosyncratic_dfs.items():
    score = get_sentiment_score(df)
    idiosyncratic_scores[ticker] = score
    print(f"{ticker} idiosyncratic sentiment: {score:.4f} ({'negative' if score < 0 else 'positive'})")

# Save
today = pd.Timestamp.today().strftime("%Y-%m-%d")

systemic_df_save = pd.DataFrame([{
    "date": today,
    "systemic_score": systemic_score
}])

idio_df_save = pd.DataFrame([{
    "date": today,
    "ticker": ticker,
    "idiosyncratic_score": score
} for ticker, score in idiosyncratic_scores.items()])

systemic_df_save.to_csv(f"{data_path}/sentiment_systemic.csv", index=False)
idio_df_save.to_csv(f"{data_path}/sentiment_idiosyncratic.csv", index=False)

print("\nSaved:")
print("  sentiment_systemic.csv")
print("  sentiment_idiosyncratic.csv")

Systemic sentiment score: -0.2767
  → negative macro backdrop

AAPL idiosyncratic sentiment: -0.3798 (negative)
MSFT idiosyncratic sentiment: -0.1556 (negative)
GOOGL idiosyncratic sentiment: -0.1711 (negative)
JPM idiosyncratic sentiment: -0.1715 (negative)
SOXX idiosyncratic sentiment: 0.1752 (positive)

Saved:
  sentiment_systemic.csv
  sentiment_idiosyncratic.csv
